# 03 — Bucle ReAct: pensar, actuar, observar

**Level 3 — Agentic AI & Workflows**

El loop que define a los agentes modernos: el agente razona, ejecuta una
tool, observa el resultado y repite hasta llegar a la respuesta. El
notebook muestra cada paso del ciclo.

In [1]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv()

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODELO = "llama3.2"
MAX_PASOS = 5


def get_hora() -> str:
    """Devuelve la hora actual."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")


def capital(texto: str) -> str:
    """Devuelve el texto en mayusculas."""
    return texto.upper()

## Tools + ejecutores

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_hora",
            "description": "Devuelve la hora actual en formato HH:MM:SS",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "capital",
            "description": "Convierte un texto a mayusculas",
            "parameters": {
                "type": "object",
                "properties": {"texto": {"type": "string"}},
                "required": ["texto"],
            },
        },
    },
]

EJECUTAR = {
    "get_hora": lambda a: get_hora(),
    "capital": lambda a: capital(a["texto"]),
}

## El loop ReAct

In [3]:
def paso_react(historial: list[dict]) -> dict:
    """Un paso del loop: pide al modelo su siguiente accion."""
    payload = {
        "model": MODELO,
        "messages": historial,
        "tools": TOOLS,
        "stream": False,
    }
    response = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]


def resolver(pregunta: str) -> str:
    """Loop ReAct: razona, usa tools y observa hasta llegar a la respuesta."""
    historial = [{"role": "user", "content": pregunta}]

    for paso in range(MAX_PASOS):
        mensaje = paso_react(historial)
        historial.append(mensaje)

        contenido = mensaje.get("content", "")
        llamadas = mensaje.get("tool_calls", [])

        if contenido:
            print(f"[paso {paso + 1}] razonamiento: {contenido.strip()}")

        if not llamadas:
            return contenido.strip()

        for llamada in llamadas:
            nombre = llamada["function"]["name"]
            argumentos = llamada["function"]["arguments"]
            if isinstance(argumentos, str):
                argumentos = json.loads(argumentos or "{}")
            resultado = EJECUTAR[nombre](argumentos)
            print(f"[paso {paso + 1}] tool {nombre}{argumentos} -> {resultado!r}")
            historial.append(
                {"role": "tool", "content": json.dumps({"resultado": resultado})}
            )

    return "No llegue a una respuesta en el maximo de pasos."

## Pregunta 1: ¿Qué hora es?

In [4]:
print("Pregunta: Que hora es?")
print("Respuesta:", resolver("Que hora es?"))

Pregunta: Que hora es?


[paso 1] tool get_hora{} -> '13:52:57'


[paso 2] razonamiento: La hora actual es las 13:52:57.
Respuesta: La hora actual es las 13:52:57.


## Pregunta 2: mayúsculas

In [5]:
print("Pregunta: Escribe la palabra hola en mayusculas")
print("Respuesta:", resolver("Escribe la palabra hola en mayusculas"))

Pregunta: Escribe la palabra hola en mayusculas


[paso 1] tool capital{'texto': 'hola'} -> 'HOLA'


[paso 2] razonamiento: La palabra "hola" escrita en mayúsculas es: HOLA.
Respuesta: La palabra "hola" escrita en mayúsculas es: HOLA.


## Conclusión

1. **Paso 1**: el modelo pidió la tool (get_hora o capital) — sin texto
2. **Paso 2**: con el resultado en el historial, razonó y respondió
3. El rol `tool` cierra el ciclo: sin él, el modelo respondería a ciegas
4. `MAX_PASOS` es la red de seguridad contra loops infinitos